# Building an MCP Server for a Database and an API

In this demo, we look at two MCP servers that give an agent structured data: 
- `ledger_server.py`, over the company's ledger in SQLite, 
- and `rates_server.py`, over the Frankfurter exchange-rate API. 

In [6]:
import sqlite3
from pathlib import Path


## 1. The Ledger tools

We will create two tools for the agent to use to query the ledger database:

- `get_schema` returns the database schema so the agent knows the tables and their structure. 
- `run_schema` runs one SQL query on the ledger and return its rows, each as a list of values.

`get_schema` returns the `CREATE TABLE` statement of every table. It reads them from `sqlite_master`, the catalog every SQLite database keeps of its own tables. This is what the agent reads before it writes any SQL.

In [ ]:
DB = Path("../data/ledger.db")


def connect() -> sqlite3.Connection:
    """Open the ledger read-only.

    mode=ro asks SQLite itself to refuse any write, so the ledger stays safe
    whatever SQL the model writes.
    """
    return sqlite3.connect(f"file:{DB}?mode=ro", uri=True)


In [8]:
def get_schema() -> str:
    """The ledger's tables, as the SQL that creates them: their columns, types and links
    to other tables. Read it before writing a query."""
    with connect() as db:
        return "\n\n".join(
            sql
            for (sql,) in db.execute(
                "SELECT sql FROM sqlite_master WHERE type = 'table'"
            )
        )


In [10]:
schema = get_schema()
print(schema)

CREATE TABLE recyclable (
	deleted_at DATETIME, 
	destroyed_at DATETIME, 
	recycled_type VARCHAR(255) NOT NULL, 
	id INTEGER NOT NULL, 
	created_at DATETIME NOT NULL, 
	updated_at DATETIME NOT NULL, 
	PRIMARY KEY (id)
)

CREATE TABLE user (
	name VARCHAR(255) NOT NULL, 
	email VARCHAR(255) NOT NULL, 
	entity_id INTEGER NOT NULL, 
	id INTEGER NOT NULL, 
	created_at DATETIME NOT NULL, 
	updated_at DATETIME NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(entity_id) REFERENCES entity (id)
)

CREATE TABLE currency (
	id INTEGER NOT NULL, 
	name VARCHAR(255) NOT NULL, 
	code VARCHAR(3) NOT NULL, 
	entity_id INTEGER NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(id) REFERENCES recyclable (id), 
	FOREIGN KEY(entity_id) REFERENCES entity (id)
)

CREATE TABLE recycled (
	recycled_id INTEGER NOT NULL, 
	restored_at DATETIME, 
	entity_id INTEGER NOT NULL, 
	id INTEGER NOT NULL, 
	created_at DATETIME NOT NULL, 
	updated_at DATETIME NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(recycled_id) REFERENCES recycl

In [12]:
print(schema.count("CREATE TABLE"), "tables\n")

14 tables



Some of these tables are python-accounting's internals, like `recyclable` or `user`. The ones we need are `account`, `transaction` and `line_item`.

## 2. Running Queries

In [13]:
def run_query(sql: str) -> list[list]:
    """Run one SQL query on the ledger and return its rows, each as a list of values.
    Args:
        sql: One SELECT statement, in SQLite syntax.
    """
    with connect() as db:
        return [list(row) for row in db.execute(sql).fetchall()]

In [14]:
run_query(
    'SELECT transaction_type, count(*) FROM "transaction" GROUP BY transaction_type'
)

[['CLIENT_INVOICE', 40], ['CLIENT_RECEIPT', 20], ['SUPPLIER_BILL', 15]]

`transaction` is a reserved word in SQL, so the table name needs quotes. Without them, the query fails:

In [15]:
try:
    run_query("SELECT count(*) FROM transaction")
except Exception as error:
    print(f"{type(error).__name__}: {error}")

OperationalError: near "transaction": syntax error


Each transaction has a client or supplier account, but its amount isn't on the transaction itself:

In [16]:
run_query(
    'SELECT transaction_no, transaction_date, main_account_amount FROM "transaction" LIMIT 3'
)

[['IN01/0001', '2026-03-23 00:00:00.000000', 0],
 ['IN01/0002', '2026-02-03 00:00:00.000000', 0],
 ['IN01/0003', '2026-02-20 00:00:00.000000', 0]]

## 3. A Write Attempt

The server opens the ledger with `mode=ro`, read-only. Let's try to delete every transaction:

In [17]:
try:
    run_query('DELETE FROM "transaction"')
except Exception as error:
    print(f"Refused: {type(error).__name__}: {error}")

run_query('SELECT count(*) FROM "transaction"')

Refused: OperationalError: attempt to write a readonly database


[[75]]

## 4. Exchange Rates

`get_rate` calls the Frankfurter API, which serves the European Central Bank's daily reference rates.

In [ ]:
import httpx

RATES_URL = "https://api.frankfurter.dev/v1"


def get_rate(date: str, from_currency: str, to_currency: str) -> dict:
    """The exchange rate between two currencies on a date. On a weekend or holiday,
    it returns the rate from the last working day before it.

    Args:
        date: The date, as YYYY-MM-DD.
        from_currency: The currency to convert from, e.g. "USD".
        to_currency: The currency to convert to, e.g. "EUR".
    """
    response = httpx.get(
        f"{RATES_URL}/{date}",
        params={"base": from_currency, "symbols": to_currency},
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    return {
        "date": data["date"],
        "from": from_currency,
        "to": to_currency,
        "rate": data["rates"][to_currency],
    }


In [21]:
get_rate("2026-09-30", "USD", "EUR")

{'date': '2026-09-30', 'from': 'USD', 'to': 'EUR', 'rate': 0.88067}

14 March 2026 is a Saturday. There are no rates on weekends, so the API returns the rate of the last working day before it, and the date it gives back tells us which day that was:

In [22]:
get_rate("2026-03-14", "USD", "EUR")

{'date': '2026-03-13', 'from': 'USD', 'to': 'EUR', 'rate': 0.87138}

## 5. The Agent

Both servers are registered in `opencode.jsonc`, and `finance-agent` can use their three tools. We run it on a question that needs both sources, and print each tool call: what the agent sent, and what came back.

In [26]:
import json
import sys

from dotenv import load_dotenv

load_dotenv(".env")
sys.path.insert(0, str(Path("../..").resolve()))
from shared.agent_runs import run_agent, summarize

QUESTION = "How much did we invoice Kestrel Media in total, in euros at the USD to EUR rate on 30 September 2026?"
run = summarize(run_agent("finance-agent", QUESTION, Path.cwd()))

for call in run["tool_calls"]:
    print(f"{call['tool']} {json.dumps(call['input'])}")
    print(f"  -> {call['output'][:300]}\n")
print("Answer:", run["answer"])
print(f"\n{run['tokens']} tokens, ${run['cost_usd']:.4f}, {run['latency_s']:.0f} s")

ledger_get_schema {}
  -> CREATE TABLE recyclable (
	deleted_at DATETIME, 
	destroyed_at DATETIME, 
	recycled_type VARCHAR(255) NOT NULL, 
	id INTEGER NOT NULL, 
	created_at DATETIME NOT NULL, 
	updated_at DATETIME NOT NULL, 
	PRIMARY KEY (id)
)

CREATE TABLE user (
	name VARCHAR(255) NOT NULL, 
	email VARCHAR(255) NOT NULL,

ledger_run_query {"sql": "SELECT transaction_type, COUNT(*) c FROM \"transaction\" GROUP BY transaction_type ORDER BY c DESC;"}
  -> [["CLIENT_INVOICE",40],["CLIENT_RECEIPT",20],["SUPPLIER_BILL",15]]

ledger_run_query {"sql": "SELECT id, transaction_date, transaction_no, narration, reference, main_account_amount, credited FROM \"transaction\" WHERE transaction_type='CLIENT_INVOICE' AND (narration LIKE '%Kestrel%' OR reference LIKE '%Kestrel%') ORDER BY transaction_date;"}
  -> [[277,"2026-01-19 00:00:00.000000","IN01/0038","Invoice for Kestrel Media",null,0,0],[212,"2026-02-10 00:00:00.000000","IN01/0030","Invoice for Kestrel Media",null,0,0],[240,"2026-04-20 00:0

## 6. Checking the Agent's Answer

We work out the answer ourselves: Kestrel Media's invoices from their line items, converted with the same rate.

In [23]:
total_usd = run_query("""
SELECT SUM(l.amount * l.quantity)
FROM "transaction" t
JOIN account a ON a.id = t.account_id
JOIN line_item l ON l.transaction_id = t.id
WHERE t.transaction_type = 'CLIENT_INVOICE' AND a.name = 'Kestrel Media'
""")[0][0]
rate = get_rate("2026-09-30", "USD", "EUR")["rate"]

print(f"Invoiced to Kestrel Media: ${total_usd:,.2f}")
print(f"USD to EUR on 30 September 2026: {rate}")
print(f"In euros: €{total_usd * rate:,.2f}")

Invoiced to Kestrel Media: $29,750.00
USD to EUR on 30 September 2026: 0.88067
In euros: €26,199.93


The agent didn't use the line items. It used the `ledger` table, where python-accounting records each invoice as double-entry postings. Let's run its last query:

In [ ]:
run_query("""
SELECT ROUND(SUM(l.amount), 4) AS total_usd
FROM ledger l
JOIN "transaction" t ON t.id = l.transaction_id
JOIN account a ON a.id = l.post_account_id
WHERE t.transaction_type = 'CLIENT_INVOICE'
  AND a.name = 'Kestrel Media'
  AND l.entry_type = 'DEBIT'
""")